# Farm Agent — 1. Tests, retrieval benchmark, yield model

**Settings:** Accelerator = GPU T4 (CPU also works, just slower) · Internet = On. No secrets needed.
**Inputs to attach:** `akshatgupta7/crop-yield-in-indian-states-dataset` (and the repo dataset, optional).

Work happens in `/tmp`; results are copied to `/kaggle/working/`:
- `pytest.txt` — test-suite output
- `retrieval/retrieval_eval.{json,md}` — dense vs BM25 vs hybrid vs hybrid+rerank
- `yield_model/` — trained yield model + `metadata.json` with its metrics

In [ ]:
REPO = "https://github.com/shashank-sj/farm-agent.git"
BRANCH = "main"
# Code source: a private Kaggle dataset with the repo (attached as input) if present, else GitHub.
import glob, os, shutil, tarfile, zipfile
DST = "/tmp/farm-agent"
shutil.rmtree(DST, ignore_errors=True)
apps = glob.glob("/kaggle/input/**/app.py", recursive=True)
archives = glob.glob("/kaggle/input/**/farm_agent_src.*", recursive=True)
if apps:
    shutil.copytree(os.path.dirname(apps[0]), DST)
elif archives:
    os.makedirs(DST)
    a = archives[0]
    (tarfile.open(a) if a.endswith((".tar.gz", ".tgz")) else zipfile.ZipFile(a)).extractall(DST)
else:
    !git clone --depth 1 -b $BRANCH $REPO $DST
%cd $DST
OUT = "/kaggle/working"

def find_input(slug, handle):
    """Directory of an attached Kaggle dataset (any mount layout); download only as a fallback."""
    for d in glob.glob("/kaggle/input/**/", recursive=True):
        if os.path.basename(os.path.normpath(d)) == slug:
            return d
    import kagglehub
    return kagglehub.dataset_download(handle)

In [ ]:
!pip install -q -r requirements-dev.txt -r requirements-ingest.txt

In [ ]:
!python -m pytest -v 2>&1 | tee $OUT/pytest.txt | tail -n 40

## Yield model (time-based split: train ≤ 2015, test ≥ 2016)

In [ ]:
path = find_input("crop-yield-in-indian-states-dataset", "akshatgupta7/crop-yield-in-indian-states-dataset")
CSV = sorted(glob.glob(f"{path}/**/*.csv", recursive=True))[0]
print(CSV)
!python -m scripts.train_yield_model --csv "$CSV" --out outputs/yield_model
!mkdir -p $OUT/yield_model && cp outputs/yield_model/* $OUT/yield_model/

## Retrieval benchmark (one query per question group against all cleaned answers, with bootstrap CIs)

In [ ]:
!python -m scripts.eval_retrieval --out $OUT/retrieval
print(open(f"{OUT}/retrieval/retrieval_eval.md").read())